# R6 牛津 Tutorial LLM 仿真 (v6.0)

## Tutorial Fellow Persona (Cell 1)

> **System Prompt (注入仿真器)**:
>
> You are an Oxford tutorial fellow in **研究伦理与AI治理 (Research ethics + AI governance: IRB, NIST RMF, EU AI Act, responsible AI, bias/fairness)**.
>
> - **Never give direct answers.** 你不直接给答案。
> - **Use Socratic questioning.** 用苏格拉底式追问。
> - **Act as HBS devil's advocate.** 扮演 HBS 魔鬼代言人, 主动反驳学生论点。
> - **Reject vague claims.** 拒绝模糊断言 (如"这个案例有点风险"), 要求引用具体 Belmont 原则 / NIST 步骤 / EU AI Act 条款号。
> - **End each turn with a probing question.** 每轮以一个追问结束。
>
> **理论依据**: Oxford tutorial (1:1 深度对话) + HBS case method (devil's advocate) + Hattie 4 级反馈 (Task/Process/Self-Reg/Feed-Forward)。
>
> **限频**: 每单元 1 次/天 (防依赖, 见 Cell 6)。本仿真用**静态 if/else 模拟** Socratic 追问, 不调 openai/anthropic API。


## Pre-Tutorial Task (Cell 2) -- 强制 Retrieval

> **开 tutorial 前学生必须先提交** (retrieval practice, 优于重读):

**提交物**: 一段 200-300 字英文 essay, 题目:

> "Pick one AI+marketing research case (e.g., AI dynamic pricing, AI personalized recommendation, AI A/B copy testing, or AI customer service). Apply Belmont Report (1979) three principles AND NIST AI RMF four-step cycle (research-ethics lens) to assess it. State the EU AI Act risk level with article number. End with one 天道推演 high-leverage point."

**要求**:
- 必须用 Belmont 三原则英文术语 (Respect for Persons / Beneficence / Justice)
- 必须用 NIST 四步英文 (Govern / Map / Measure / Manage) 且明确"研究伦理视角"
- 必须带 EU AI Act 条款号 (Article 5 / Annex III / Article 50)
- 必须给出一个可执行的高杠杆点 (小投入改变大局)

**为什么先写 essay**: Oxford tutorial 假定学生已做 retrieval, 否则 tutorial 沦为讲授。Hattie (2009) meta-analysis 显示 pre-task retrieval 提升 effect size d=0.6+。本仿真在 Cell 3 假定学生 essay 已提交, 仿真器基于 essay 内容追问。

**学生 essay (示例输入, 写在下面变量 `student_essay` 中)**:


In [ ]:
# Cell 3: Multi-turn Socratic Loop (>=4 rounds, 静态 if/else 模拟, 不调 API)
# 含 >=5 个苏格拉底问: 为什么/反例/若前提变/凭什么/如何
# 领域: Belmont + NIST AI RMF + EU AI Act + garak/PyRIT + 天道推演

import json
from pathlib import Path

# --- 学生 essay (retrieval 产物, 此处用示例占位, 真实使用时学生自己填) ---
student_essay = '''
Case: AI dynamic pricing research on elderly users without informed consent.
Belmont: Respect violated (no consent), Beneficence questionable (max revenue not max benefit), Justice violated (elderly bear risk).
NIST: Govern (no IRB), Map (elderly identified as vulnerable), Measure (no bias audit), Manage (no opt-out).
EU AI Act: prohibited under Article 5 (manipulation + vulnerable group exploitation).
High-leverage point: add informed consent + differential privacy before deployment.
'''

print("=" * 70)
print("STUDENT ESSAY SUBMITTED:")
print("=" * 70)
print(student_essay)
print()

# --- 静态 Socratic 仿真器 (if/else 模拟 5 类追问, 不调 LLM API) ---
# 5 类苏格拉底问: 为什么 (why) / 反例 (counterexample) / 若前提变 (what-if) / 凭什么 (on what grounds) / 如何 (how)

def socratic_tutor(essay_text, round_num):
    """静态模拟牛津 tutor 的第 N 轮追问 (不调 API)。返回 (追问, 期望学生回答方向)。"""
    round_num = int(round_num)
    if round_num == 1:
        # Q1: 为什么 -- 挑战因果
        return (
            "[Round 1 / WHY] You claim Respect is violated because 'no consent'. "
            "WHY is consent the operative criterion here, rather than Beneficence? "
            "In other words, why is the consent gap a Respect problem and not primarily a Beneficence gap? "
            "Cite the Belmont Report definition.",
            "期望: 学生应区分 Respect=自主权/知情同意 vs Beneficence=风险-收益最大化, 引用 Belmont 原文"
        )
    elif round_num == 2:
        # Q2: 反例 -- HBS devil's advocate
        return (
            "[Round 2 / COUNTEREXAMPLE] Devil's advocate: suppose the AI dynamic pricing "
            "actually LOWERS prices for elderly users in 70%% of cases (a benefit). "
            "Does this counterexample weaken your Justice claim? If the elderly benefit on average, "
            "is the burden still unfairly distributed? Refute or concede.",
            "期望: 学生应承认分布公平≠平均收益, 即使 70% 受益, 30% 受损的弱势群体仍构成 Justice 违反 (Barocas & Selbst 2016 部署偏见)"
        )
    elif round_num == 3:
        # Q3: 若前提变 -- 反事实推演
        return (
            "[Round 3 / WHAT-IF] If the research team ADDS informed consent tomorrow "
            "(your high-leverage point), does the EU AI Act risk level change? "
            "Does it drop from 'prohibited (Article 5)' to 'high risk (Annex III)'? "
            "Or does Article 5 still apply because the vulnerable-group exploitation persists? "
            "Trace the counterfactual.",
            "期望: 学生应指出 Article 5 禁止性条款基于行为类型 (操控+弱势剥削), 非仅知情同意; 加同意不够, 需停止弱势群体定向定价"
        )
    elif round_num == 4:
        # Q4: 凭什么 -- 证据/条款号
        return (
            "[Round 4 / ON WHAT GROUNDS] You write 'NIST: Govern (no IRB)'. "
            "On what grounds do you map 'IRB approval' to Govern rather than to Map? "
            "The NIST AI RMF 1.0 Core page lists Govern functions including 'accountability structures' "
            "and Map functions including 'context establishment'. Which clause specifically supports "
            "your mapping? Quote the function description.",
            "期望: 学生应引用 NIST AI RMF 1.0 Core: Govern=问责结构/政策流程 (含 IRB 审批制度), Map=上下文/参与者识别 (含人类参与者识别)"
        )
    else:
        # Q5: 如何 -- 可执行性
        return (
            "[Round 5 / HOW] Your high-leverage point is 'add informed consent + differential privacy'. "
            "HOW exactly would you implement differential privacy in a dynamic pricing model "
            "without destroying price signal accuracy? Which epsilon? And how would garak/PyRIT "
            "red-team probes (dan/promptinject/goodside) verify the Beneficence claim after DP is added? "
            "Walk through the verification chain.",
            "期望: 学生应给出 epsilon 选择 (如 epsilon=1.0), garak probe 在 DP 后重跑, 比较 pre/post 漏洞数, 验证 Beneficence 改善"
        )

# --- 跑 5 轮 Socratic loop ---
for r in range(1, 6):
    question, expected = socratic_tutor(student_essay, r)
    print(f"{'=' * 70}")
    print(f"TUTOR (Round {r}):")
    print(f"{'=' * 70}")
    print(question)
    print()
    print(f"[期望回答方向 (内部, 不展示给学生)]: {expected}")
    print()
    print("[学生在下方 mental sandbox 模拟回答 -> 下一轮追问基于上一轮 expected 方向静态推进]")
    print()

# 说明: 真实牛津 tutorial 中, tutor 会基于学生实际回答动态调整; 本仿真用静态 if/else
# 按预定义 5 类追问顺序推进, 覆盖 why/counterexample/what-if/on-what-grounds/how。
print("=" * 70)
print("5 轮 Socratic loop 结束。进入 Cell 4 student_model.json 读写。")
print("=" * 70)


In [ ]:
# Cell 4: student_model.json 读写 (记录掌握度/盲点)
# 基于 Hattie formative feedback 的 Self-Reg 级: 学生自我追踪盲点

import json
from pathlib import Path

student_model = {
    "unit": "R6",
    "student_id": "phd_student_001",
    "timestamp": "2026-07-26T10:00:00",
    "ilo_mastery": {
        "ILO1_Belmont_principles": {
            "score": 0.85,
            "attempts": 2,
            "mastered": True,
            "last_blind_spot": "混淆 Respect 子要求'自主权'为独立原则 (diagnostic D1)"
        },
        "ILO2_pydantic_IRB_scorer": {
            "score": 0.72,
            "attempts": 3,
            "mastered": False,
            "last_blind_spot": "score_to_status 阈值边界 (compliant>=80 vs >80) 与 weakest_principle argmin 漂移"
        },
        "ILO3_NIST_research_lens": {
            "score": 0.65,
            "attempts": 2,
            "mastered": False,
            "last_blind_spot": "NIST Map vs Govern 混淆 (IRB审批应映射 Govern 问责结构, 非 Map 上下文)"
        },
        "ILO4_EU_AI_Act_articles": {
            "score": 0.80,
            "attempts": 2,
            "mastered": True,
            "last_blind_spot": None
        },
        "ILO5_redteam_tiandao": {
            "score": 0.55,
            "attempts": 1,
            "mastered": False,
            "last_blind_spot": "天道推演 far 层因果断链 (罚款/禁令/声誉未与 near 层 GDPR 调查因果相连)"
        }
    },
    "diagnostic_pretest": {
        "D1_belmont_misconception": "passed_after_retry",
        "D2_NIST_lens_confusion": "failed_once",
        "D3_EU_article5_vs_annex3": "passed"
    },
    "weak_loop_triggered": {
        "drill_3_consecutive_fails": 2,
        "fallback_action": "回退 drill-3 阶段3 -> 阶段2 Faded + 补充 solution.ipynb TODO6 worked example",
        "status": "in_progress"
    },
    "recommended_review_units": ["R6 (本单元 drill-3 重做)", "S2 (模块S 合规, NIST 企业视角对照)"]
}

# 写入 student_model.json
out_path = Path("student_model.json")
out_path.write_text(json.dumps(student_model, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"[WROTE] {out_path.resolve()}")

# 读回验证
loaded = json.loads(out_path.read_text(encoding="utf-8"))
print(f"[READ BACK] unit={loaded['unit']}, mastered_ILOs={sum(1 for v in loaded['ilo_mastery'].values() if v['mastered'])}/5")
print(f"[WEAK LOOP] drill-3 连续失败 {loaded['weak_loop_triggered']['drill_3_consecutive_fails']} 次, 触发回退: {loaded['weak_loop_triggered']['fallback_action']}")
print()
print("盲点汇总 (mastery<0.8 的 ILO):")
for ilo, info in loaded["ilo_mastery"].items():
    if not info["mastered"]:
        print(f"  - {ilo}: score={info['score']}, blind_spot={info['last_blind_spot']}")


## Cell 5: Hattie 4 级 Formative Feedback

> Hattie & Timperley (2007) 4 级反馈。**避免 Self 级表扬** (如"你真聪明"), 改用 Self-Reg (学生自我追踪盲点)。


In [ ]:
# Cell 5 (code): Hattie 4 级反馈生成器
# 4 个标记: [TASK] / [PROCESS] / [SELF-REG] / [FEED-FORWARD]
# 避免 [SELF] 表扬级 (Hattie: Self 级 d=0.14 弱效, Self-Reg 级 d=0.69 强效)

feedback = {
    "[TASK]": (
        "Task-level (反馈具体任务表现): "
        "你的 essay 正确识别了 AI 动态定价在 Belmont Respect (no informed consent) 上的违反, "
        "且 EU AI Act 判定为 Article 5 禁止 -- 这两项 AT 命中。 "
        "但 NIST 映射中 'Govern (no IRB)' 表述模糊: IRB 审批制度映射 Govern 的问责结构 (accountability structures), "
        "而非简单的 'no IRB'。修正: 'Govern: 缺失 IRB 审批制度与研究者问责结构 (NIST AI RMF 1.0 Core GOVERN 2.1)。'"
    ),
    "[PROCESS]": (
        "Process-level (反馈解题策略): "
        "你的解题策略是 'Belmont -> NIST -> EU' 线性。更优策略是 **三视角交叉印证**: "
        "对同一案例同时跑 Belmont 评分 + NIST 步骤 + EU 条款, 检查三者短板是否一致 (如 Belmont Respect 低 + NIST Measure 偏弱 + EU Article 5 -> 三视角同指向'参与者保护失败')。 "
        "这个交叉策略在 practice.md drill-2 feedback_rule 中强制要求三元组输出。"
    ),
    "[SELF-REG]": (
        "Self-Regulation-level (反馈学生自我监控, 非 Self 表扬): "
        "你在 student_model.json 中自评 ILO5=0.55 (红队+天道推演), 且正确识别盲点 'far 层因果断链'。 "
        "这是有效的 Self-Reg: 你知道自己哪里不会。下一步: 触发 weak_loop (drill-3 连续 2 次失败), "
        "回退阶段3 -> 阶段2 Faded, 并手抄 solution.ipynb TODO6 worked example, 重做后 self-Reg 重新评分。 "
        "注意: 不要停留在'我伦理学得不好'这类 Self 级自我评价 (d=0.14 弱效), 聚焦 Self-Reg (d=0.69)。"
    ),
    "[FEED-FORWARD]": (
        "Feed-Forward-level (下一步去哪): "
        "基于盲点, 推荐复习路径: "
        "(1) R6 drill-3 阶段2 Faded 重做 (天道推演 far 层因果连接); "
        "(2) 模块S (合规) Day1 NIST AI RMF 企业治理视角 -- 与本单元研究伦理视角对照, 消除 ILO3 Map/Govern 混淆; "
        "(3) OECD AI Incidents Monitor 取 2 个新事件, 独立跑三视角交叉印证, 作为 progressive_project milestone 的补充案例。"
    )
}

for level, text in feedback.items():
    print(f"{'=' * 70}")
    print(f"{level}")
    print(f"{'=' * 70}")
    print(text)
    print()

# 自检: 4 个标记齐全且无 [SELF] 表扬级
markers = ["[TASK]", "[PROCESS]", "[SELF-REG]", "[FEED-FORWARD]"]
present = [m for m in markers if m in feedback]
forbidden_self_praise = ["聪明", "好棒", "well done", "great job", "smart"]
has_self_praise = any(p in " ".join(feedback.values()).lower() for p in forbidden_self_praise)
print(f"[SELF-CHECK] 4 markers present: {len(present)}/4 -> {present}")
print(f"[SELF-CHECK] Self-level praise detected: {has_self_praise} (must be False)")
assert len(present) == 4, "缺标记"
assert not has_self_praise, "检测到 Self 级表扬, 违反 Hattie 原则"
print("[OK] Hattie 4 级反馈自检通过。")


## Cell 6: 限频 + Exit Artifact

### 限频 (防依赖)

- **每单元 1 次/天**: 本 Oxford tutorial LLM 仿真每天最多 1 次, 防止学生把 tutorial 当成"答案生成器"依赖。
- **理由**: Oxford tutorial 的价值在于学生**先做 retrieval** (Cell 2 essay), 再接受 Socratic 追问。若每天多次, 学生会跳过 retrieval 直接问 tutor, 丧失 effect size d=0.6+ 的 retrieval gain。
- **强制机制**: 在 `student_model.json` 中记录 `last_tutorial_date`, 若当天已用过 -> 拒绝执行 Cell 3, 提示"明日再来, 今日先做 retrieval"。
- **例外**: 若 weak_loop 触发 (连续 2 次失败), 允许追加 1 次 (用于 Hattie [SELF-REG] 反馈后重做)。

### Exit Artifact (tutorial 结束必交)

> Tutorial 结束后, 学生必须提交以下 exit artifact (写入 `student_model.json` 的 `exit_artifact` 字段):

1. **2-3 个盲点** (基于 5 轮 Socratic 追问 + Hattie [TASK]/[PROCESS] 反馈, 自我识别):
   - 例: "我把 NIST IRB 映射到 Map 而非 Govern -- 混淆了'上下文识别'与'问责结构'"
   - 例: "我假设加知情同意就能从 Article 5 降到 Annex III -- 忽略了 Article 5 基于行为类型而非同意"
   - 例: "我的天道推演 far 层 (罚款) 与 near 层 (GDPR 调查) 因果断链 -- 没说明 GDPR 调查如何导致罚款"

2. **推荐复习单元** (基于盲点, 至少 1 个本单元 + 1 个跨单元):
   - 本单元: `practice.md` drill-3 阶段2 Faded 重做 (天道推演因果链)
   - 跨单元: 模块S (合规) Day1 -- NIST AI RMF 企业治理视角, 与本单元研究伦理视角对照
   - 跨模块: `data/README.md` 中 OECD AI Incidents Monitor, 取 2 个新事件做三视角交叉印证

3. **下一步 action** (可执行, 24h 内完成):
   - 例: "今晚 22:00 前完成 drill-3 阶段2 Faded (AI 自动文案 A/B 测试 far 层填空), 提交到 student_model.json"

### Tutorial 闭环

```
retrieval (Cell 2 essay) -> Socratic 5 轮 (Cell 3) -> student_model 读写 (Cell 4)
  -> Hattie 4 级反馈 (Cell 5) -> exit artifact (Cell 6) -> 下次 retrieval (明天)
```

**理论依据**: Oxford tutorial (1:1 深度对话) + HBS devil's advocate + Hattie 4 级反馈 + 限频防依赖。Effect size (Hattie 2009): tutorial 类 d=0.55, Self-Reg 反馈 d=0.69, retrieval d=0.6+。

---

*本 tutorial.ipynb 用 Python 生成器脚本 (md/code/nb helpers + json.dump ensure_ascii=False indent=1) 写出。Socratic loop 为静态 if/else 模拟, 不调 openai/anthropic API。所有追问领域特定, 引用 Belmont/NIST/EU AI Act/garak/PyRIT/天道推演真实框架。*
